# Análise Exploratória — Roubos de Veículos no Rio de Janeiro

Este notebook prototiipa e valida o pipeline de dados antes da produção final.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.etl import load_raw, clean, summary_stats

## 1. Carregamento e limpeza

In [ ]:
df_raw = load_raw()
print('Formato bruto:', df_raw.shape)
df_raw.head()

In [ ]:
df = clean(df_raw)
print('Formato processado:', df.shape)
df.head()

## 2. Estatísticas descritivas

In [ ]:
summary_stats(df)

## 3. RQ1 — Evolução absoluta de roubos

In [ ]:
fig = px.line(
    df, x='ano', y='incidencias',
    markers=True,
    title='RQ1 — Roubos de veículos no 1º quadrimestre (2003–2022)',
    labels={'ano': 'Ano', 'incidencias': 'Incidências'}
)
fig.show()

## 4. RQ2 — Taxa por 100 mil habitantes e variação percentual

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=df['ano'], y=df['taxa_100k_hab'], name='Taxa/100k hab', marker_color='steelblue'))
fig.add_trace(go.Scatter(x=df['ano'], y=df['dif_percentual'], name='Var. % a.a.', yaxis='y2', mode='lines+markers', line=dict(color='crimson')))
fig.update_layout(
    title='RQ2 — Taxa por 100 mil hab. e variação percentual anual',
    yaxis=dict(title='Taxa por 100k hab.'),
    yaxis2=dict(title='Variação % a.a.', overlaying='y', side='right'),
    legend=dict(x=0.01, y=0.99)
)
fig.show()

## 5. RQ3 — Média mensal por período

In [ ]:
fig = px.bar(
    df, x='ano', y='media_mensal',
    color='media_mensal', color_continuous_scale='Reds',
    title='RQ3 — Média mensal de roubos no 1º quadrimestre',
    labels={'ano': 'Ano', 'media_mensal': 'Média mensal'}
)
fig.show()

## 6. Exportar gráficos para docs/assets

In [ ]:
import plotly.io as pio
from pathlib import Path

assets = Path('../docs/assets')
assets.mkdir(parents=True, exist_ok=True)

fig_abs = px.line(df, x='ano', y='incidencias', markers=True,
    title='Roubos de veículos — 1º quadrimestre (2003–2022)',
    labels={'ano': 'Ano', 'incidencias': 'Incidências'})
pio.write_html(fig_abs, file=str(assets / 'fig_rq1_evolucao.html'), include_plotlyjs='cdn', full_html=False)

fig_taxa = go.Figure()
fig_taxa.add_trace(go.Bar(x=df['ano'], y=df['taxa_100k_hab'], name='Taxa/100k hab', marker_color='steelblue'))
fig_taxa.add_trace(go.Scatter(x=df['ano'], y=df['dif_percentual'], name='Var. % a.a.', yaxis='y2', mode='lines+markers', line=dict(color='crimson')))
fig_taxa.update_layout(
    title='Taxa por 100 mil hab. e variação percentual anual',
    yaxis=dict(title='Taxa por 100k hab.'),
    yaxis2=dict(title='Variação % a.a.', overlaying='y', side='right')
)
pio.write_html(fig_taxa, file=str(assets / 'fig_rq2_taxa.html'), include_plotlyjs='cdn', full_html=False)

fig_media = px.bar(df, x='ano', y='media_mensal', color='media_mensal', color_continuous_scale='Reds',
    title='Média mensal de roubos — 1º quadrimestre',
    labels={'ano': 'Ano', 'media_mensal': 'Média mensal'})
pio.write_html(fig_media, file=str(assets / 'fig_rq3_media.html'), include_plotlyjs='cdn', full_html=False)

print('Figuras exportadas para docs/assets/')